# musikoekosysteme-wien_scripts

Verifiziert die zentralen Zahlen des Berichts "Musikalische Ökosysteme in Wien" gegen die Quell-CSVs in `daten/`. Jeder Abschnitt entspricht einem Kapitel des Berichts.

Vor jeder neuen Sitzung zuerst den Abschnitt "Setup" ausführen.

## Setup

In [ ]:
import os, json
import pandas as pd
import numpy as np
import plotly.express as px
from plotly.subplots import make_subplots

print("Arbeitsverzeichnis:", os.getcwd())
print("Inhalt von daten/:", os.listdir('daten') if os.path.exists('daten') else "daten/ nicht gefunden")

In [ ]:
def zu_zahl(s):
    return pd.to_numeric(s, errors='coerce')

def vereine_laden():
    v = pd.read_csv('daten/vereine.csv', sep='\t', dtype=str)
    v['Betrag_EUR_num'] = zu_zahl(v['Betrag_EUR'])
    v['zaehlbar'] = ~v['Fördergegenstand — Anmerkungen'].str.contains('— Ausnahme', regex=False, na=False)
    return v

def bezirkskultur_laden():
    vb = pd.read_csv('daten/foerderungen_bezirkskultur.csv', sep='\t', dtype=str)
    vb['Betrag_EUR_num'] = zu_zahl(vb['Betrag_EUR'])
    return vb

v = vereine_laden()
vb = bezirkskultur_laden()
print(f"vereine.csv: {len(v)} Zeilen ({v['zaehlbar'].sum()} zählbar, ohne Ausnahme)")
print(f"foerderungen_bezirkskultur.csv: {len(vb)} Zeilen")

## 1. KONTEXT
2023-Gesamtsumme aus MA 7, BMKöS und Bezirkskultur (4.799.600 Euro laut Bericht).

In [ ]:
ma7_2023 = v[(v['Foerderung_ref']=='MA7') & (v['Jahr']=='2023') & (v['zaehlbar'])]['Betrag_EUR_num'].sum()
bmkos_2023 = v[(v['Foerderung_ref']=='BMKöS') & (v['Jahr']=='2023') & (v['zaehlbar'])]['Betrag_EUR_num'].sum()
bk_2023 = vb[vb['Jahr']=='2023']['Betrag_EUR_num'].sum()
summe_2023 = ma7_2023 + bmkos_2023 + bk_2023

print(f"MA 7 2023: {ma7_2023:,.0f} | BMKöS 2023: {bmkos_2023:,.0f} | Bezirkskultur 2023: {bk_2023:,.0f}")
print(f"Summe: {summe_2023:,.0f} (Bericht: 4.799.600)")

## 3.1 Verteilung der Förderung nach Quelle
MA 7 / BMKöS / BMWKMS je Jahr, Bezirkskultur je Jahr, Wien Modern 2023-Konzentration.

In [ ]:
for quelle in ['MA7', 'BMKöS', 'BMWKmS']:
    print(f"--- {quelle} ---")
    print(v[(v['Foerderung_ref']==quelle) & (v['zaehlbar'])].groupby('Jahr')['Betrag_EUR_num'].sum())
    print()

print("--- Bezirkskultur ---")
print(vb.groupby('Jahr')['Betrag_EUR_num'].sum())
print(f"Gesamt: {vb['Betrag_EUR_num'].sum():,.2f} (Bericht: 419.034,59)")

In [ ]:
wm_2023 = v[(v['Verein'].str.contains('Wien Modern', case=False, na=False)) & (v['Jahr']=='2023') & (v['zaehlbar']) & (v['Foerderung_ref']=='MA7')]['Betrag_EUR_num'].sum()
ma7_2023_gesamt = v[(v['Foerderung_ref']=='MA7') & (v['Jahr']=='2023') & (v['zaehlbar'])]['Betrag_EUR_num'].sum()
print(f"Wien Modern 2023 MA7: {wm_2023:,.0f} EUR ({wm_2023/ma7_2023_gesamt*100:.2f}% von MA7 2023, Bericht: 40,97%)")

## 3.2 Historische MA 7 Reihe
Prozentuale Veränderung Jahr für Jahr.

In [ ]:
ma7_reihe = v[(v['Foerderung_ref']=='MA7') & (v['zaehlbar'])].groupby('Jahr')['Betrag_EUR_num'].sum()
print(ma7_reihe)
print()
for a, b in zip(ma7_reihe.index[:-1], ma7_reihe.index[1:]):
    veraenderung = (ma7_reihe[b] - ma7_reihe[a]) / ma7_reihe[a] * 100
    print(f"{a}->{b}: {veraenderung:.2f}%")

## 3.3 Räumliche Verteilung: Förderung nach Bezirk
Tabelle 2, Konzentration der drei größten Bezirke.

In [ ]:
tabelle_bezirk = v[(v['Foerderung_ref']=='MA7') & (v['zaehlbar'])].groupby('Bezirk')['Betrag_EUR_num'].sum().sort_values(ascending=False)
print(tabelle_bezirk)
print()
top3 = tabelle_bezirk.head(3).sum()
gesamt = tabelle_bezirk.sum()
print(f"Top 3 Bezirke: {top3:,.0f} von {gesamt:,.0f} = {top3/gesamt*100:.2f}% (Bericht: 64,63%)")

In [ ]:
korpus = pd.read_csv('daten/kombinierter_korpus.csv', sep='\t', dtype=str)
korpus['Bezirk_norm'] = korpus['Bezirk'].astype(str).str.strip().str.replace(r'\.0$', '', regex=True)
anzahl = korpus.groupby('Bezirk_norm').size().sort_values(ascending=False)
print(anzahl.head(10))
print()
print("Bezirk 3:", anzahl.get('3'), "(Bericht: 64)")
print("Bezirk 6:", anzahl.get('6'), "(Bericht: 268)")

b6_eur = v[(v['Bezirk']=='6') & (v['Foerderung_ref']=='MA7') & (v['zaehlbar'])]['Betrag_EUR_num'].sum()
print("Bezirk 6, MA7 EUR:", b6_eur, "(Bericht: 38.000)")

## 3.4 Programme und Förderung
Eindeutige Programme, Kontinuität ≤2 Jahre vs. ≥3 Jahre. Vereinheitlichung von Schreibvarianten
und Ausschluss generischer Bezeichnungen (z.B. "Halbjahr"-Sammelposten) sind bereits enthalten.

In [ ]:
programme = v[(v['zaehlbar']) & (v['Foerderung_ref'].isin(['MA7','BMKöS'])) & (v['Programm'].notna())].copy()

generisches_muster = r'Halbjahr|^Veranstaltungsreihe$|^Projektreihe$'
programme = programme[~programme['Programm'].str.contains(generisches_muster, regex=True, na=False)]

programm_zuordnung = {
    'This is our dancefloor! 2023': 'This is our Dancefloor',
    'TENDER MATTER Konzertreihe': 'Tender Matter, The Future Konzertreihe',
    'Kanal 41.3': 'Kanal #41.2',
    'NAWA music & film festival 2022': 'nawa music & film Festival',
    'nawa music & film festival': 'nawa music & film Festival',
    'NAWA FESTIVAL – music and film festival': 'nawa music & film Festival',
    'Sonic Territories Festival': 'Sonic Territories',
    'Konzert Veranstaltungsreihe ((liccht))': 'Veranstaltungsreihe „liccht"',
    'Black Page Orchestra Konzerte': 'Black Page Orchestra',
    'Konzertzyklus 3 Konzerte im REAKTOR': 'Konzertzyklus im REAKTOR',
    'Club-Eventreihe Augend&Addend – Undoing the equation': 'Augend&Addend – Undoing the equation',
    'Projekt: HYPERREALITY - Festival für Club Culture - Vienna 2023': 'Projekt: HYPERREALITY - Festival für Club Culture - Vienna',
    'Projekt: HYPERREALITY - Festival für Club Culture - Vienna 2024': 'Projekt: HYPERREALITY - Festival für Club Culture - Vienna',
    'Festival SUAL 2023': 'Festival SUAL',
    'Unsafe+Sounds Festival 2023': 'Unsafe+Sounds Festival',
    'Monday Improvisers Session.': 'Monday Improvisers Session',
    'Salam Orient 2022': 'Salam Orient',
}
programme['Programm'] = programme['Programm'].replace(programm_zuordnung)

anzahl_programme = programme['Programm'].nunique()
zusammenfassung = programme.groupby('Programm').agg(EUR=('Betrag_EUR_num','sum'), n_jahre=('Jahr','nunique'))
bis_2_jahre = (zusammenfassung['n_jahre'] <= 2).sum()
ab_3_jahre = (zusammenfassung['n_jahre'] >= 3).sum()

print(f"Programme: {anzahl_programme} (Bericht: 127)")
print(f"<=2 Jahre: {bis_2_jahre} ({bis_2_jahre/anzahl_programme*100:.1f}%, Bericht: 110, 86,6%)")
print(f">=3 Jahre: {ab_3_jahre} ({ab_3_jahre/anzahl_programme*100:.1f}%, Bericht: 17, 13,4%)")

## Bekannte manuelle Entscheidungspunkte
Diese Fälle wurden im Projektverlauf gefunden und erfordern menschliches Urteil, kein Skript
löst sie automatisch. Bei einer künftigen Aktualisierung der Quelldaten hier erneut prüfen.

- **HYPERREALITY**: zwei Zeilen ("HYPERREALITY", 18.000 EUR, 2022-2023, und "Projekt: HYPERREALITY - Festival für Club Culture - Vienna", 260.000 EUR, 2022-2024). Nicht automatisch zusammengeführt, da unklar ist, ob es sich um dasselbe Programm handelt.
- **czirp czirp / IMPROPER WALLS**: mehrere Zeilen mit identischem Jahr und Betrag sind in der Regel *keine* Duplikate, sondern unterschiedliche Förderlinien (Musik, Stadtteilkultur etc.). Vor jeder Deduplizierung nach Verein+Jahr+Betrag zusätzlich `Fördergegenstand — Anmerkungen` (mit `.str.strip()`) einbeziehen, sonst werden echte, unterschiedliche Förderzeilen fälschlich als Duplikat entfernt.
- **Text mit `\n`**: einige Zeilen in `Fördergegenstand — Anmerkungen` und `Programm` enthielten unsichtbare Zeilenumbrüche am Ende, wodurch inhaltlich identische Einträge nicht als solche erkannt wurden. Vor jedem Duplikat- oder Gruppierungs-Check: `.str.replace(r'\n', ' ', regex=True).str.replace(r'\s+', ' ', regex=True).str.strip()`.